In [68]:
import polars as pl

In [69]:
def show_all(df, width=200, max_col_width=True):
    '''
    Prints an entire polars dataframe in the console or notebook output.
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width of the printed dataframe.
        Defaults to 200.
    max_col_width : bool, optional
        Whether to set the maximum column width.
        i.e. it will print the full contents of the cells.
        Defaults to True.
    '''
    with  pl.Config()  as  cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        if  max_col_width  or  len(df.columns) ==  1:
            cfg.set_fmt_str_lengths(width)
        print(df)

# batch 8

In [70]:
import extern
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt >/tmp/runs")

''

In [71]:
prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
prev.shape

(63406, 1)

In [72]:
total = pl.read_csv('runlists/acc_less_than_20k_mbytes.csv', has_header=False)
total.columns = ["acc"]
total.shape

(710928, 1)

In [73]:
batch8_possible = total.join(prev, on="acc", how="anti")
batch8_possible.shape

(647522, 1)

In [74]:
# batch8 = batch8_possible.sample(100000)
# print(batch8.shape, batch8[:5])
# batch8.write_csv('runlists/batch8_100000.txt', include_header=False)

In [75]:
# 
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt runlists/batch8_100000.txt >/tmp/runs")

prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
print(prev.shape)

batch9_possible = total.join(prev, on="acc", how="anti")
batch9_possible.shape

(163406, 1)


(547522, 1)

In [76]:
# batch9_possible.write_csv('runlists/batch9_da_rest.txt', include_header=False)

# batch10 - cleaning up and everythin 10Gbp or less

In [77]:
extern.run('cat  s3_us-east2.txt s3_woodcrob-sandpiper-us-east-1.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev1 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev1.columns = ["acc", "time", "size", "path"]
prev2 = prev1.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0)).select(["acc", "size"])
prev2.shape, prev2[:3]

((137852, 2),
 shape: (3, 2)
 ┌───────────┬───────┐
 │ acc       ┆ size  │
 │ ---       ┆ ---   │
 │ str       ┆ i64   │
 ╞═══════════╪═══════╡
 │ DRR001961 ┆ 26759 │
 │ DRR003630 ┆ 60975 │
 │ DRR003636 ┆ 45914 │
 └───────────┴───────┘)

In [78]:
prev2.filter(pl.col("size") == 0).shape

(4305, 2)

In [79]:
prev3 = prev2.filter(pl.col("size") > 0)
prev3.shape

(133547, 2)

In [80]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [81]:
df.shape, sum(df['mbases'])/1e9
#=> 780k metagenomes, 4.9 Pbases. A lot.

((783205, 4), 4.866907013)

In [82]:
batch10_possible = df.filter(pl.col('mbases') < 8000).join(prev, on="acc", how="anti")
batch10_possible.shape, batch10_possible.sample(3)

((471178, 4),
 shape: (3, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ ERR13967733 ┆ 2024-11-21T00:00:00+00:00 ┆ 5348   ┆ 1588   │
 │ SRR31375808 ┆ 2024-11-18T00:00:00+00:00 ┆ 1358   ┆ 435    │
 │ SRR29763399 ┆ 2024-07-10T00:00:00+00:00 ┆ 3621   ┆ 1170   │
 └─────────────┴───────────────────────────┴────────┴────────┘)

In [83]:
batch10_possible.select('acc').write_csv('runlists/batch10_10gbp_or_smaller.txt', include_header=False)

# batch11 small fast test of multiple

In [84]:
per_acc = pl.read_csv('~/m/msingle/mess/174_R220_renew/processing_20240531/per_acc_summary.csv')
per_acc.shape, per_acc[:3]

((245436, 11),
 shape: (3, 11)
 ┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
 │ sample     ┆ root_cove ┆ species_c ┆ bacterial ┆ … ┆ warning ┆ predictio ┆ organism  ┆ host_or_n │
 │ ---        ┆ rage      ┆ overage   ┆ _archaeal ┆   ┆ ---     ┆ n         ┆ ---       ┆ ot        │
 │ str        ┆ ---       ┆ ---       ┆ _bases    ┆   ┆ str     ┆ ---       ┆ str       ┆ ---       │
 │            ┆ f64       ┆ f64       ┆ ---       ┆   ┆         ┆ str       ┆           ┆ str       │
 │            ┆           ┆           ┆ i64       ┆   ┆         ┆           ┆           ┆           │
 ╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
 │ SRR8634435 ┆ 345.4     ┆ 0.933237  ┆ 117232064 ┆ … ┆ null    ┆ host      ┆ feces met ┆ host      │
 │            ┆           ┆           ┆ 2         ┆   ┆         ┆           ┆ agenome   ┆           │
 │ SRR8640623 ┆ 730.5     ┆ 0.127748  ┆ 139594647 ┆

In [85]:
batch11 = batch10_possible.join(per_acc.filter(pl.col('root_coverage')>0), left_on='acc', right_on='sample', how='inner').sort('mbases').head(4)

In [86]:
# batch11.select('acc').write_csv('runlists/batch11_4.txt', include_header=False)

# batch 12 - 8 to 30Gbp

In [87]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [88]:
# how much 0-8, 8-15, 15-30 and 30+ mbases?
(
    df.select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 15000).filter(pl.col('mbases') >= 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 15000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') >= 30000).select('mbases').sum()[0,0] /1e9,
)


(4.866907013, 1.763913074, 1.36664454, 0.898161387, 0.838188012)

In [89]:
# Try those < 8gbases again where they failed - maybe the increased RAM of a this batch will get them through?
extern.run('cat s3_ls/* |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev4 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev4.columns = ["acc", "time", "size", "path"]
print(prev4.group_by(pl.col('size') > 0).len())
prev5 = prev4.filter(pl.col('size') > 0).with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))
prev5.shape

shape: (2, 2)
┌───────┬────────┐
│ size  ┆ len    │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 8769   │
│ true  ┆ 591446 │
└───────┴────────┘


(591446, 4)

In [90]:
# Create a batch of 1000 runs with 8-30 Gbps, that aren't in any previous 
batch12_possible = df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 8000).join(prev5, on="acc", how="anti")
batch12_possible.shape[0], batch12_possible.sample(5), batch12_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(146031,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR28841769 ┆ 2024-04-30T00:00:00+00:00 ┆ 21768  ┆ 6236   │
 │ SRR25947381 ┆ 2023-09-07T00:00:00+00:00 ┆ 11294  ┆ 3508   │
 │ SRR27449297 ┆ 2024-01-07T00:00:00+00:00 ┆ 14901  ┆ 4487   │
 │ SRR28523231 ┆ 2024-06-02T00:00:00+00:00 ┆ 8223   ┆ 2629   │
 │ SRR16817059 ┆ 2022-11-05T00:00:00+00:00 ┆ 18562  ┆ 5852   │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 1.957546801)

In [91]:
# batch12_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch12_8_to_30gbp.txt', include_header=False)

# batch 13 - 30gbp and above

In [92]:
batch13_possible = df.filter(pl.col('mbases') >= 30000).join(prev5, on="acc", how="anti")
batch13_possible.shape[0], batch13_possible.sample(5), batch13_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(16011,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ ERR12651941 ┆ 2024-02-10T00:00:00+00:00 ┆ 33523  ┆ 10333  │
 │ ERR10921321 ┆ 2023-09-06T00:00:00+00:00 ┆ 45469  ┆ 13811  │
 │ SRR25400521 ┆ 2023-07-24T00:00:00+00:00 ┆ 53866  ┆ 17181  │
 │ SRR17794289 ┆ 2022-01-28T00:00:00+00:00 ┆ 39687  ┆ 13162  │
 │ ERR10101704 ┆ 2022-08-21T00:00:00+00:00 ┆ 61143  ┆ 17809  │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 0.798983057)

In [93]:
# batch13_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch13_30gbp_plus.txt', include_header=False)

# batch 14 - 30gbp and above retrying failed ones

In [94]:
# ➜  date; rclone ls s3:woodcrob-sandpiper-us-east-1/unannotated6/ >unannotated6.finished.ls.txt
# Mon 24 Mar 2025 09:27:23 AEST
# ➜  wc -l unannotated6.finished.ls.txt
#14262 unannotated6.finished.ls.txt
extern.run('cat  unannotated6.finished.ls.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')
batch13_finished_samples = pl.read_csv("/tmp/runs", has_header=False, separator='\t')
batch13_finished_samples.columns = ["null","size","acc1"]
batch13_finished_samples.drop_in_place("null")
batch13_finished_samples = batch13_finished_samples.with_columns(pl.col("acc1").str.split('.').list.get(0).alias("acc"))
# batch13_finished_samples.columns = ["acc"]
batch13_finished_samples.shape, batch13_finished_samples[:4], batch13_finished_samples.filter(pl.col("size") == 0).shape

((14262, 3),
 shape: (4, 3)
 ┌─────────┬─────────────────────────────────┬───────────┐
 │ size    ┆ acc1                            ┆ acc       │
 │ ---     ┆ ---                             ┆ ---       │
 │ i64     ┆ str                             ┆ str       │
 ╞═════════╪═════════════════════════════════╪═══════════╡
 │ 2925000 ┆ DRR046408.unannotated.singlem.… ┆ DRR046408 │
 │ 5731055 ┆ DRR046817.unannotated.singlem.… ┆ DRR046817 │
 │ 0       ┆ DRR046818.unannotated.singlem.… ┆ DRR046818 │
 │ 2753842 ┆ DRR046819.unannotated.singlem.… ┆ DRR046819 │
 └─────────┴─────────────────────────────────┴───────────┘,
 (1325, 3))

In [95]:
batch14_possible = batch13_possible.join(batch13_finished_samples.filter(pl.col("size") != 0), on="acc", how="anti")
batch14_possible.shape, batch14_possible[:4]

((3075, 4),
 shape: (4, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR5451739  ┆ 2017-04-17T00:00:00+00:00 ┆ 57300  ┆ 25291  │
 │ SRR9008757  ┆ 2019-05-04T00:00:00+00:00 ┆ 68571  ┆ 20772  │
 │ SRR23272262 ┆ 2024-02-28T00:00:00+00:00 ┆ 149473 ┆ 47852  │
 │ SRR28417531 ┆ 2024-03-22T00:00:00+00:00 ┆ 66738  ┆ 21106  │
 └─────────────┴───────────────────────────┴────────┴────────┘)

In [96]:
# show_all(batch14_possible.sample(10))
batch14_possible.select(pl.col('mbytes').sum()) / 1e6
# => 68 TB. Bit much to do locally.

mbytes
f64
67.970189


In [97]:
# batch14_possible.sample(fraction=1).select('acc').write_csv('runlists/batch14_30gbp_plus_retries.txt', include_header=False)

# Batch 15 - chunking into 3 Gbp chunks

In [98]:
# ben in 🌐 b2 in sandpiper/sra_metadata/shotgun_sra_20250220 on  main [!?] via singlem-dev 20250324 took 48s 
# ➜  jq -rc '[.acc,.releasedate,.mbases,.mbytes,.organism,.avgspotlen, (.attributes[] | select(.k == "bases") | .v)] |@csv' <(cat *) |pigz >../sra_metadata_20250220.some_columns3.csv.gz
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns3.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes','organism','avespotlen','bases']
df[:4]

acc,releasedate,mbases,mbytes,organism,avespotlen,bases
str,str,i64,i64,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614,"""viral metagenome""",300,6638665200
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195,"""feces metagenome""",300,8555841630
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344,"""gut metagenome""",299,1013578651
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792,"""metagenome""",281,20619243718


In [101]:
# Let's go batch14_possible, split into 3Gbp chunks
# Use the bases column because it is much more accurate than mbases
batch15_possible = batch14_possible.join(df.select('acc','avespotlen','bases'), on="acc", how="inner")
batch15_possible = batch15_possible.with_columns((pl.col('mbases') / 1e3).alias('gbases'))
gbp_per_chunk = 3
batch15_possible = batch15_possible.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
# batch16_possible = batch16_possible.with_columns((pl.col('gbases') / gbp_per_chunk).alias('num_chunks'))
batch15_possible = batch15_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
batch15_possible.shape, batch15_possible[:4]

((3075, 9),
 shape: (4, 9)
 ┌─────────────┬─────────────┬────────┬────────┬───┬────────────┬─────────┬────────────┬────────────┐
 │ acc         ┆ releasedate ┆ mbases ┆ mbytes ┆ … ┆ bases      ┆ gbases  ┆ chunk_size ┆ num_chunks │
 │ ---         ┆ ---         ┆ ---    ┆ ---    ┆   ┆ ---        ┆ ---     ┆ ---        ┆ ---        │
 │ str         ┆ str         ┆ i64    ┆ i64    ┆   ┆ i64        ┆ f64     ┆ i32        ┆ i32        │
 ╞═════════════╪═════════════╪════════╪════════╪═══╪════════════╪═════════╪════════════╪════════════╡
 │ SRR5451739  ┆ 2017-04-17T ┆ 57300  ┆ 25291  ┆ … ┆ 5730089170 ┆ 57.3    ┆ 9933775    ┆ 20         │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 4          ┆         ┆            ┆            │
 │             ┆ :00         ┆        ┆        ┆   ┆            ┆         ┆            ┆            │
 │ SRR9008757  ┆ 2019-05-04T ┆ 68571  ┆ 20772  ┆ … ┆ 6857192913 ┆ 68.571  ┆ 9933775    ┆ 23         │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 6 

In [102]:
# Ensure that avespotlen * chunk_size * num_chunks is > bases
batch15_possible.filter(pl.col('chunk_size') * pl.col('avespotlen') * pl.col('num_chunks') < pl.col('bases'))

acc,releasedate,mbases,mbytes,avespotlen,bases,gbases,chunk_size,num_chunks
str,str,i64,i64,i64,i64,f64,i32,i32


In [103]:
batch15_possible_exploded = batch15_possible.group_by('acc','avespotlen','num_chunks','gbases','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')


In [104]:
# batch15_possible_exploded.sample(fraction=1, shuffle=True).write_csv('runlists/batch15_3gbp_chunks.txt')

# batch 16 - 15gbp chunks

In [105]:
# Let's go batch14_possible, split into 3Gbp chunks
# Use the bases column because it is much more accurate than mbases
batch16_possible = batch14_possible.join(df.select('acc','avespotlen','bases'), on="acc", how="inner")
batch16_possible = batch16_possible.with_columns((pl.col('mbases') / 1e3).alias('gbases'))
gbp_per_chunk = 15
batch16_possible = batch16_possible.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
batch16_possible = batch16_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
batch16_possible.shape, batch16_possible[:4]

((3075, 9),
 shape: (4, 9)
 ┌─────────────┬─────────────┬────────┬────────┬───┬────────────┬─────────┬────────────┬────────────┐
 │ acc         ┆ releasedate ┆ mbases ┆ mbytes ┆ … ┆ bases      ┆ gbases  ┆ chunk_size ┆ num_chunks │
 │ ---         ┆ ---         ┆ ---    ┆ ---    ┆   ┆ ---        ┆ ---     ┆ ---        ┆ ---        │
 │ str         ┆ str         ┆ i64    ┆ i64    ┆   ┆ i64        ┆ f64     ┆ i32        ┆ i32        │
 ╞═════════════╪═════════════╪════════╪════════╪═══╪════════════╪═════════╪════════════╪════════════╡
 │ SRR5451739  ┆ 2017-04-17T ┆ 57300  ┆ 25291  ┆ … ┆ 5730089170 ┆ 57.3    ┆ 49668875   ┆ 4          │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 4          ┆         ┆            ┆            │
 │             ┆ :00         ┆        ┆        ┆   ┆            ┆         ┆            ┆            │
 │ SRR9008757  ┆ 2019-05-04T ┆ 68571  ┆ 20772  ┆ … ┆ 6857192913 ┆ 68.571  ┆ 49668875   ┆ 5          │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 6 

In [108]:
batch16_possible_exploded = batch16_possible.group_by('acc','avespotlen','num_chunks','gbases','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')
batch16_possible_exploded[:10], batch16_possible_exploded.shape

(shape: (10, 6)
 ┌─────────────┬────────────┬────────────┬────────┬────────────┬───────┐
 │ acc         ┆ avespotlen ┆ num_chunks ┆ gbases ┆ chunk_size ┆ chunk │
 │ ---         ┆ ---        ┆ ---        ┆ ---    ┆ ---        ┆ ---   │
 │ str         ┆ i64        ┆ i32        ┆ f64    ┆ i32        ┆ i64   │
 ╞═════════════╪════════════╪════════════╪════════╪════════════╪═══════╡
 │ SRR22308412 ┆ 300        ┆ 3          ┆ 34.971 ┆ 50000000   ┆ 1     │
 │ SRR22308412 ┆ 300        ┆ 3          ┆ 34.971 ┆ 50000000   ┆ 2     │
 │ SRR22308412 ┆ 300        ┆ 3          ┆ 34.971 ┆ 50000000   ┆ 3     │
 │ SRR8861596  ┆ 302        ┆ 3          ┆ 42.517 ┆ 49668875   ┆ 1     │
 │ SRR8861596  ┆ 302        ┆ 3          ┆ 42.517 ┆ 49668875   ┆ 2     │
 │ SRR8861596  ┆ 302        ┆ 3          ┆ 42.517 ┆ 49668875   ┆ 3     │
 │ ERR3346067  ┆ 149        ┆ 6          ┆ 89.997 ┆ 100671141  ┆ 1     │
 │ ERR3346067  ┆ 149        ┆ 6          ┆ 89.997 ┆ 100671141  ┆ 2     │
 │ ERR3346067  ┆ 149        ┆ 6    

In [109]:
batch16_possible_exploded.sample(fraction=1, shuffle=True).write_csv('runlists/batch16_15gbp_chunks.txt')